# Fairness Monitoring Module — Demo

End-to-end demonstration using the **loan approval dataset**.

**Sensitive attribute:** `Gender`  
**Target:** `Loan_Approval_Status`  
**Model:** Logistic Regression (trained here from scratch)

Pipeline:
1. Load data & train model
2. `RealTimeFairnessTracker` — simulate production stream in batches
3. `FairnessDriftAndAlertEngine` — detect fairness drift
4. `FairnessReportingDashboard` — visualize & report
5. `FairnessABTestAnalyzer` — compare baseline vs fair model


## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from monitoring.tracker    import RealTimeFairnessTracker
from monitoring.drift      import FairnessDriftAndAlertEngine
from monitoring.reporting  import FairnessReportingDashboard
from monitoring.ab_testing import FairnessABTestAnalyzer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports OK")


---
## 1. Load Data & Train Model


In [ ]:
df = pd.read_csv("../data/loan_dataset.csv")
df.dropna(inplace=True)
print(f"Dataset shape : {df.shape}")
print(f"Approval rate : {df['Loan_Approval_Status'].mean():.3f}")
print(f"Gender counts :\n{df['Gender'].value_counts()}")


In [ ]:
FEATURE_COLS = ["Age", "Annual_Income", "Credit_Score",
                "Loan_Amount_Requested", "Employment_Status"]
TARGET_COL   = "Loan_Approval_Status"
SENSITIVE    = "Gender"

df_enc = pd.get_dummies(df[FEATURE_COLS + [TARGET_COL, SENSITIVE]], drop_first=True)

X = df_enc.drop(columns=[TARGET_COL])
y = df_enc[TARGET_COL]
gender_col = [c for c in X.columns if "Gender" in c][0]  # e.g. "Gender_Male"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)
y_proba = model.predict_proba(X_test_sc)[:, 1]

print(classification_report(y_test, y_pred))


In [ ]:
# Recover Gender for the test set (0 = Female, 1 = Male based on get_dummies drop_first)
gender_binary = X_test[gender_col].values  # 1 = Male, 0 = Female

print("Approval rate by gender (predictions):")
for g, label in [(1, "Male"), (0, "Female")]:
    mask = gender_binary == g
    print(f"  {label}: {y_pred[mask].mean():.3f}  (n={mask.sum()})")


---
## 2. RealTimeFairnessTracker

We simulate a **production stream** by shuffling the test set and
splitting it into 25 equal batches ingested one at a time.
Each batch represents one hour of production traffic.


In [ ]:
# Shuffle test set and split into batches
N_BATCHES  = 25
rng        = np.random.default_rng(RANDOM_STATE)
idx        = rng.permutation(len(y_test))

predictions_all = y_pred[idx]
labels_all      = y_test.values[idx]
sensitive_all   = gender_binary[idx]

batch_size = len(idx) // N_BATCHES
batches = [
    {
        "predictions": predictions_all[i*batch_size:(i+1)*batch_size],
        "labels":      labels_all     [i*batch_size:(i+1)*batch_size],
        "sensitive":   sensitive_all  [i*batch_size:(i+1)*batch_size],
    }
    for i in range(N_BATCHES)
]

print(f"Total test samples : {len(idx)}")
print(f"Batches            : {N_BATCHES}")
print(f"Samples per batch  : {batch_size}")


In [ ]:
tracker = RealTimeFairnessTracker(window_size=8, min_group_size=10)

BASE_TS = datetime(2024, 6, 1, 9, 0)

for i, batch in enumerate(batches):
    ts = BASE_TS + timedelta(hours=i)
    tracker.ingest(**batch, timestamp=ts, sensitive_name="gender")

history = tracker.history
print(f"History shape : {history.shape}")
print(f"Columns       : {list(history.columns)}")
history.tail(6)


In [ ]:
print("=== Latest window snapshot ===")
print(tracker.latest()[["gender", "demographic_parity", "equalized_odds",
                         "positive_rate", "group_size"]]
      .to_string(index=False))


---
## 3. FairnessDriftAndAlertEngine

Runs a KS test to detect drift and classifies it as `spike` or `trend`
via wavelet decomposition. Alerts are prioritized as `CRITICAL / HIGH / LOW`.


In [ ]:
engine = FairnessDriftAndAlertEngine(
    reference_window=8,
    ks_alpha=0.05,
    adaptive=True,
    group_size_weight=True,
)

alerts = engine.analyze(history)
print(f"Alerts generated : {len(alerts)}")
for a in alerts:
    print(f"  [{a.severity:8s}] {a.metric:25s}  "
          f"KS={a.drift_statistic:.3f}  p={a.p_value:.4f}  "
          f"score={a.severity_score:.3f}  wavelet={a.wavelet_trend}")


In [ ]:
engine.alert_summary()


In [ ]:
# Adaptive threshold status
if engine.threshold_manager is not None:
    print("Current adaptive thresholds:")
    print(engine.threshold_manager.summary().to_string(index=False))


---
## 4. FairnessReportingDashboard


In [ ]:
dashboard = FairnessReportingDashboard(title="Loan Approval — Fairness Monitor")


### 4a. Demographic parity over time

In [ ]:
fig = dashboard.trend_plot(
    history,
    metric="demographic_parity",
    group_col="gender",
    threshold=engine.get_threshold("demographic_parity"),
    alert_timestamps=[a.timestamp for a in alerts if a.metric == "demographic_parity"],
    title="Demographic Parity Gap Over Time (Male vs Female)",
)
fig.show()


### 4b. Positive rate by gender

In [ ]:
fig = dashboard.trend_plot(
    history,
    metric="positive_rate",
    group_col="gender",
    title="Approval Rate by Gender Over Time",
)
fig.show()


### 4c. Intersectional bar chart

In [ ]:
fig = dashboard.intersectional_plot(
    history,
    metric="positive_rate",
    attr1_col="gender",
    recent_n=8,
    title="Mean Approval Rate by Gender (last 8 batches)",
)
fig.show()


### 4d. Multi-metric summary

In [ ]:
fig = dashboard.summary_plot(
    history,
    metrics=["demographic_parity", "equalized_odds", "predictive_parity"],
    group_col="gender",
)
fig.show()


### 4e. Markdown report

In [ ]:
path = dashboard.generate_report(
    history=history,
    alerts=alerts,
    output_path="fairness_report.md",
    group_col="gender",
)
with open(path) as f:
    print(f.read())


---
## 5. FairnessABTestAnalyzer

We compare:
- **Control** — baseline logistic regression (no fairness constraint)
- **Treatment** — model trained with class weights to partially mitigate gender bias

This simulates evaluating whether a fairness intervention actually helped.


In [ ]:
# Treatment model: class_weight='balanced' as a simple fairness intervention
model_fair = LogisticRegression(
    max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
)
model_fair.fit(X_train_sc, y_train)
y_pred_fair = model_fair.predict(X_test_sc)

print("=== Baseline model ===")
print(f"  Overall accuracy : {(y_pred == y_test.values).mean():.4f}")
print(f"  Female approval  : {y_pred[gender_binary==0].mean():.4f}")
print(f"  Male approval    : {y_pred[gender_binary==1].mean():.4f}")
print()
print("=== Fair model (balanced weights) ===")
print(f"  Overall accuracy : {(y_pred_fair == y_test.values).mean():.4f}")
print(f"  Female approval  : {y_pred_fair[gender_binary==0].mean():.4f}")
print(f"  Male approval    : {y_pred_fair[gender_binary==1].mean():.4f}")


In [ ]:
# Build DataFrames for the analyzer
ctrl_df = pd.DataFrame({
    "prediction" : y_pred,
    "label"      : y_test.values,
    "gender"     : np.where(gender_binary == 1, "Male", "Female"),
    # Continuous columns for mediation
    "score"      : y_proba,
    "approval_rate": y_pred.astype(float),
})

trt_df = pd.DataFrame({
    "prediction" : y_pred_fair,
    "label"      : y_test.values,
    "gender"     : np.where(gender_binary == 1, "Male", "Female"),
    "score"      : model_fair.predict_proba(X_test_sc)[:, 1],
    "approval_rate": y_pred_fair.astype(float),
})

analyzer = FairnessABTestAnalyzer(
    control=ctrl_df,
    treatment=trt_df,
    pred_col="prediction",
    label_col="label",
    sensitive_cols=["gender"],
    alpha=0.05,
)


### 5a. Statistical power per subgroup

In [ ]:
power_df = analyzer.calculate_power(effect_size=0.05, metric="accuracy")
print("=== Power Analysis (can we detect a 5% accuracy difference per gender?) ===")
power_df


### 5b. Heterogeneous treatment effects

In [ ]:
hte_df = analyzer.heterogeneous_effects(
    business_metric="accuracy",
    fairness_metric="positive_rate",
)
print("=== Did the fair model help each gender equally? ===")
hte_df


### 5c. Mediation analysis

In [ ]:
# Does the intervention work by changing approval rates (mediator),
# or does it have a direct effect on the outcome score?
mediation = analyzer.mediation_analysis(
    outcome_col="score",
    mediator_col="approval_rate",
    treatment_indicator_col="treatment",
)

print("=== Baron-Kenny Mediation Analysis ===")
print(f"  Total effect        : {mediation['total_effect']:+.4f}  (p={mediation['p_total']:.4f})")
print(f"  Direct effect       : {mediation['direct_effect']:+.4f}  (p={mediation['p_direct']:.4f})")
print(f"  Indirect effect     : {mediation['indirect_effect']:+.4f}")
prop = mediation['proportion_mediated']
if not (isinstance(prop, float) and prop != prop):  # nan check
    print(f"  Proportion mediated : {prop:.1%}")
else:
    print(f"  Proportion mediated : N/A")


---
## Summary

| Component | What was demonstrated |
|---|---|
| `RealTimeFairnessTracker` | 25-batch production stream from real loan test data, sliding window metrics per gender |
| `FairnessDriftAndAlertEngine` | KS-test drift detection + wavelet classification on real model predictions |
| `FairnessReportingDashboard` | Trend plots, approval rate by gender, intersectional bar chart, Markdown report |
| `FairnessABTestAnalyzer` | Baseline vs balanced-weight model: power analysis, HTE per gender, mediation |
